# CogniFusion — Colab Notebook

**Real-time cognitive state monitoring** that fuses EEG (Muse-style 4-channel)
and eye-tracking signals to classify **low / medium / high** cognitive load.

| | |
|---|---|
| Repo | [PoornaSri26/CogniFusion](https://github.com/PoornaSri26/CogniFusion) |
| Runtime | CPU works — pick **GPU** (Runtime → Change runtime type) to train faster |
| Data | Synthetic 30-subject dataset generated in-notebook; no hardware needed |

## What this notebook does

1. **Setup** — clone repo, install deps
2. **Data** — generate 360 synchronized EEG+eye epochs (30 subjects × 3 classes)
3. **Features** — extract 168 fused features per epoch (137 EEG + 31 eye)
4. **Explore** — signal plots, feature correlations, class separation
5. **Train** — `AttentionFusionNet`: vector-gated cross-modal attention fusion,
   focal loss, mixup augmentation, per-subject median/MAD normalization,
   temperature scaling, EMA smoothing, optional LightGBM stacking
6. **Evaluate** — *every model in the ensemble* scored individually, then the
   full ensemble, on subjects the model has never seen

## Reading the results

- **Subject-wise holdout** = the honest metric. Test subjects never appear in
  training, so it measures generalization to new people — not memorization.
- Synthetic data is clean and class-separable, so expect high scores here;
  real Muse+webcam data is harder (see the repo README for real-data results).

## 1 · Setup — clone repo & install deps

In [ ]:
# Clone the repository
!git clone -q https://github.com/PoornaSri26/CogniFusion.git
%cd CogniFusion

# Core ML deps + lightgbm for the stacked ensemble.
# torch is preinstalled on Colab — don't reinstall it.
!pip install -q numpy pandas scipy scikit-learn pyarrow lightgbm

import sys, torch
sys.path.insert(0, "src")
sys.path.insert(0, "data")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 2 · Generate the synthetic dataset

`data/simulate_data.py` synthesizes EEG (4 channels @ 256 Hz, 4 s epochs) and
eye-tracking (60 Hz: pupil, gaze, fixation, blinks) for 30 subjects × 4 epochs ×
3 classes = **360 epochs**.

Each subject gets unique biological parameters — individual alpha frequency
(9–12 Hz), noise floor, electrode offsets, pupil baseline, blink rate — so the
model must learn *subject-invariant* load features rather than memorizing noise.

In [ ]:
from pathlib import Path
from simulate_data import generate_dataset

if Path("data/raw/manifest.csv").exists():
    print("data/raw/ already populated — skipping generation")
    import pandas as pd
    manifest = pd.read_csv("data/raw/manifest.csv")
else:
    manifest = generate_dataset("data/raw")
manifest.head()


## 3 · Build the fused feature table

`src/build_dataset.py` runs feature extraction on every epoch and joins EEG +
eye features into one parquet table:

- **EEG** (`eeg_features.py`): band powers (delta/theta/alpha/beta/gamma),
  differential entropy, frontal/parietal ratios, spectral entropy, Hjorth
  params, coherence & phase-lag connectivity, fractal dimension
- **Eye** (`eye_features.py`): pupil diameter stats & variability, fixation
  ratio, blink rate, gaze dispersion/speed/acceleration, microsaccades

> This is the slowest step (~10 min on CPU) — extraction is per-epoch.
>
> **Already have data?** Drop your own `manifest.csv` + raw epochs into
> `data/raw/` (or a ready `features.parquet` into `data/processed/`) and this
> cell — like data generation above — skips straight to using it.

In [ ]:
from pathlib import Path
import pandas as pd

FEAT = Path("data/processed/features.parquet")
if FEAT.exists():
    print("features.parquet already exists — skipping feature extraction")
    df = pd.read_parquet(FEAT)
else:
    from build_dataset import build
    df = build("data/raw/manifest.csv", str(FEAT))
df.shape


## 3b · Preprocessing & data-quality checks

Before modeling: missing values, outliers (IQR fences), and per-class
feature drift. `build_dataset_real.py` already median-imputes NaNs; this cell
shows you *what* needed cleaning — run it on `features_real.parquet` after
section 12 to audit the real data.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

FEAT = "data/processed/features_real.parquet" if Path("data/processed/features_real.parquet").exists() else "data/processed/features.parquet"
df = pd.read_parquet(FEAT)
feat_cols = [c for c in df.columns if c.startswith(("eeg__","eye__"))]
print(f"Table: {FEAT}  |  {df.shape[0]} epochs x {len(feat_cols)} features")

# 1) Missing values
nan_per_col = df[feat_cols].isna().sum()
print(f"
Columns with NaNs: {(nan_per_col>0).sum()} | total cells: {int(nan_per_col.sum())}")
if (nan_per_col>0).any():
    display(nan_per_col[nan_per_col>0].sort_values(ascending=False).head(10).to_frame("NaN count"))

# 2) Outliers via IQR fences
q1, q3 = df[feat_cols].quantile([0.25, 0.75])
iqr = q3 - q1
outlier_mask = (df[feat_cols] < (q1 - 1.5*iqr)) | (df[feat_cols] > (q3 + 1.5*iqr))
outlier_rate = outlier_mask.mean().sort_values(ascending=False)
print(f"
Top-10 columns by outlier rate (>1.5xIQR):")
display((outlier_rate.head(10)*100).round(2).to_frame("% rows outlying"))
print(f"Median outlier rate across features: {outlier_rate.median()*100:.2f}%")

# 3) Class-conditional drift on top features
load = df["label"].map({"low":0,"medium":1,"high":2})
corr = df[feat_cols].corrwith(load).abs().sort_values(ascending=False)
top5 = corr.head(5).index
fig, axes = plt.subplots(1, len(top5), figsize=(3*len(top5), 3), sharey=False)
for ax, col in zip(axes, top5):
    for lab in ["low","medium","high"]:
        v = df.loc[df["label"]==lab, col].dropna()
        ax.hist(v, bins=25, alpha=0.55, label=lab, density=True)
    ax.set_title(col.replace("eeg__","").replace("eye__",""), fontsize=8)
    ax.tick_params(labelsize=7)
axes[0].legend(fontsize=7)
plt.suptitle("Class-conditional distributions — top 5 load-correlated features", fontsize=10)
plt.tight_layout(); plt.show()

# 4) Optional: clip extreme outliers (uncomment to apply)
# df[feat_cols] = df[feat_cols].clip(lower=q1-3*iqr, upper=q3+3*iqr)

## 4 · Dataset overview

In [ ]:
import pandas as pd

df = pd.read_parquet("data/processed/features.parquet")
print("Label distribution:")
display(df["label"].value_counts().to_frame("count"))

feat_cols = [c for c in df.columns if c.startswith(("eeg__", "eye__"))]
print(f"\n{len(feat_cols)} fused features | missing values: {int(df[feat_cols].isna().sum().sum())}")

load = df["label"].map({"low": 0, "medium": 1, "high": 2})
top = df[feat_cols].corrwith(load).abs().sort_values(ascending=False).head(10)
print("\nTop features by |correlation| with cognitive load:")
display(top.to_frame("|r|"))

## 5 · Signal explorer — what does load look like in the raw signals?

One epoch per class: EEG time series + power spectral density (1–45 Hz),
then pupil diameter and gaze position. Expect alpha to drop and theta/beta to
rise as load increases.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import welch

manifest = pd.read_csv("data/raw/manifest.csv")
CHANNELS = ["AF7", "AF8", "TP9", "TP10"]

fig, axes = plt.subplots(3, 3, figsize=(17, 9))
for col, label in enumerate(["low", "medium", "high"]):
    row = manifest[manifest["label"] == label].iloc[0]
    eeg = np.load(row["eeg_path"])                      # (4, 1024)
    t = np.arange(eeg.shape[1]) / 256.0

    for ch in range(4):
        axes[0, col].plot(t, eeg[ch], lw=0.5, label=CHANNELS[ch])
    axes[0, col].set_title(f"EEG time series — {label}")
    axes[0, col].legend(fontsize=6, ncol=2)

    for ch in range(4):
        f, p = welch(eeg[ch], fs=256, nperseg=256)
        axes[1, col].semilogy(f, p, lw=0.8, label=CHANNELS[ch])
    axes[1, col].set_xlim(1, 45); axes[1, col].set_xlabel("Hz")
    axes[1, col].set_title("EEG power spectral density")
    axes[1, col].legend(fontsize=6, ncol=2)

    eye = pd.read_csv(row["eye_path"])
    axes[2, col].plot(eye["t"], eye["pupil_mm"], lw=0.6, color="steelblue", label="pupil mm")
    ax2 = axes[2, col].twinx()
    ax2.plot(eye["t"], eye["gaze_x"], lw=0.4, alpha=0.6, color="darkorange", label="gaze x")
    axes[2, col].set_title(f"Eye — pupil & gaze ({label})")
    axes[2, col].set_xlabel("s")
plt.tight_layout(); plt.show()

## 6 · Feature landscape — which features carry the load signal?

Left: top features ranked by correlation with load. Right: class means for the
top 6 — watch theta/beta rise and alpha fall as load increases.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

df = pd.read_parquet("data/processed/features.parquet")
feat_cols = [c for c in df.columns if c.startswith(("eeg__", "eye__"))]
load = df["label"].map({"low": 0, "medium": 1, "high": 2})

corr = df[feat_cols].corrwith(load)
top12 = corr.abs().sort_values(ascending=False).head(12).index

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
corr.loc[top12].sort_values().plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Top 12 features — correlation with cognitive load")
axes[0].set_xlabel("Pearson r"); axes[0].axvline(0, color="k", lw=0.6)

top6 = corr.abs().sort_values(ascending=False).head(6).index
df.groupby("label")[list(top6)].mean().T.plot(kind="bar", ax=axes[1],
                                             figsize=(12, 5), colormap="viridis")
axes[1].set_title("Class means — top 6 features"); axes[1].set_ylabel("mean value")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## 7 · Train the model

`src/train.py` trains `AttentionFusionNet` with:

- **Vector-gated cross-modal fusion** — per-dimension trust weighting between
  EEG and eye embeddings (the model learns *when* to trust each modality)
- **Focal loss** (γ=2) targeting the ambiguous medium class
- **Mixup augmentation**, label smoothing, modality dropout
- **Per-subject median/MAD normalization** — removes subject baselines so the
  model generalizes to unseen people
- **Temperature scaling + EMA smoothing** — calibrated, jitter-free predictions
- **LightGBM stacking** (`--use-gbm`) — 60% NN + 40% GBM probability blend

Single subject-wise holdout by default. Add `--cv 5` for 5-fold CV (~5× slower).

In [ ]:
# 5 folds x 200 epochs - use a GPU runtime for reasonable speed.
# On CPU, drop to --cv 0 (single holdout) or --epochs 100.
!python src/train.py --features data/processed/features.parquet --epochs 200 --cv 5 --use-gbm --out models/fusion_model


In [ ]:
import torch, json
ckpt = torch.load("models/fusion_model.pt", map_location="cpu", weights_only=False)
print(json.dumps(ckpt["cv_summary"], indent=2))
print("\nPer-fold validation scores stored in checkpoint:")
for i, rec in enumerate(ckpt["folds"]):
    print(f"  fold {i+1}: acc={rec['val_accuracy']:.3f}  "
          f"macro-F1={rec['val_macro_f1']:.3f}  "
          f"lgb={'yes' if rec.get('lgb_model') else 'no'}")

## 8 · Training curves

Loss & validation macro-F1 every 10 epochs, recorded in the checkpoint.

In [ ]:
import torch
import matplotlib.pyplot as plt

ckpt = torch.load("models/fusion_model.pt", map_location="cpu", weights_only=False)
for i, rec in enumerate(ckpt["folds"]):
    h = rec.get("history") or []
    if not h:
        print(f"Fold {i+1}: no history recorded — retrain to enable curves")
        continue
    ep   = [d["epoch"] for d in h]
    loss = [d["loss"]   for d in h]
    f1   = [d["val_f1"] for d in h]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep, loss, marker="o", ms=3, color="steelblue")
    ax[0].set_title(f"Fold {i+1} — training loss"); ax[0].set_xlabel("epoch")
    ax[1].plot(ep, f1, marker="o", ms=3, color="seagreen")
    ax[1].axhline(0.9, ls="--", c="r", alpha=0.4, label="target 0.9")
    ax[1].set_title("validation macro-F1"); ax[1].set_xlabel("epoch")
    ax[1].legend(); plt.tight_layout(); plt.show()

## 9 · Every model in the ensemble — scored individually, then pooled

With `--cv 5` the checkpoint holds **5 folds × (NN + LightGBM)** = 10 models.
This cell reproduces each fold's *own* held-out subject split (no leakage —
a fold is only scored on subjects it never trained on), then pools all
predictions for the headline CV score.

Per fold it reports **NN-only vs NN+GBM-stacked** accuracy, so you can see
exactly what the LightGBM blend buys you.

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

from model import AttentionFusionNet
from train import (subject_wise_split, subject_kfold, adaptive_subject_norm,
                   coerce_numeric, make_norm_key, TemperatureScaler)

def load_ensemble(path="models/fusion_model.pt"):
    # Load the self-contained ensemble checkpoint (same logic as
    # realtime_pipeline.load_model, without hardware deps like brainflow).
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    folds = []
    for rec in ckpt["folds"]:
        m = AttentionFusionNet(
            eeg_dim=len(ckpt["eeg_cols"]),
            eye_dim=len(ckpt["eye_cols"]),
            embed_dim=rec.get("embed_dim", ckpt.get("embed_dim", 48)),
            n_classes=len(ckpt["label_classes"]),
            gate_type=ckpt.get("gate_type", "vector"),
        )
        m.load_state_dict(rec["model_state"])
        m.eval()
        ts = TemperatureScaler()
        ts.load_state_dict(rec["temp_scaler"])
        ts.eval()
        folds.append({"model": m, "scaler_eeg": rec["scaler_eeg"],
                      "scaler_eye": rec["scaler_eye"], "temp": ts,
                      "lgb": rec.get("lgb_model")})
    return {"folds": folds, "eeg_cols": ckpt["eeg_cols"],
            "eye_cols": ckpt["eye_cols"],
            "label_classes": list(ckpt["label_classes"])}

bundle = load_ensemble("models/fusion_model.pt")
eeg_cols, eye_cols, classes = bundle["eeg_cols"], bundle["eye_cols"], bundle["label_classes"]

df = pd.read_parquet("data/processed/features.parquet")
df["norm_key"] = make_norm_key(df)

# --- Rebuild the SAME splits training used (k = number of folds) ---
k = len(bundle["folds"])
if k > 1:
    splits = list(subject_kfold(df, k=k, seed=42))
    print(f"{k}-fold subject-wise CV — each fold scored on its own unseen subjects")
else:
    splits = [subject_wise_split(df, test_frac=0.25, seed=42)]
    print("Single subject-wise holdout split")

def ema(raw):
    out = np.zeros_like(raw)
    out[0] = raw[0]
    for t in range(1, len(raw)):
        out[t] = 0.65 * raw[t] + 0.35 * out[t - 1]
    return out

rows, pooled = [], {"y": [], "pred": [], "conf": [], "subject": [], "label": [],
                    "epoch_id": [], "probs": []}
with torch.no_grad():
    for i, fold in enumerate(bundle["folds"]):
        _, te = splits[i]
        te = te.copy()
        te["norm_key"] = make_norm_key(te)
        y_te = te["label"].to_numpy()

        eeg_n, _ = adaptive_subject_norm(coerce_numeric(te, eeg_cols), te["norm_key"])
        eye_n, _ = adaptive_subject_norm(coerce_numeric(te, eye_cols), te["norm_key"])
        xe = np.nan_to_num(fold["scaler_eeg"].transform(eeg_n))
        ye = np.nan_to_num(fold["scaler_eye"].transform(eye_n))

        # NN-only probabilities (temperature-calibrated)
        logits = fold["model"](torch.tensor(xe, dtype=torch.float32),
                               torch.tensor(ye, dtype=torch.float32))
        logits = fold["temp"](logits)
        nn_p = torch.softmax(logits, dim=1).numpy()

        # Stacked: 60% NN + 40% LightGBM (same blend as realtime_pipeline)
        stack_p = nn_p
        if fold["lgb"] is not None:
            lgb_p = fold["lgb"].predict_proba(np.concatenate([xe, ye], axis=1))
            stack_p = 0.60 * nn_p + 0.40 * lgb_p

        for tag, p in (("nn", nn_p), ("nn+gbm", stack_p)):
            pred_i = np.array(classes)[p.argmax(1)]
            if tag == "nn+gbm":
                rows.append({"fold": i + 1,
                             "nn_acc": round(accuracy_score(y_te, np.array(classes)[nn_p.argmax(1)]), 4),
                             "nn_f1":  round(f1_score(y_te, np.array(classes)[nn_p.argmax(1)], average="macro"), 4),
                             "stack_acc": round(accuracy_score(y_te, pred_i), 4),
                             "stack_f1":  round(f1_score(y_te, pred_i, average="macro"), 4)})

        # pooled predictions use the stacked probs + EMA
        pe = ema(stack_p)
        pooled["y"] += list(y_te)
        pooled["pred"] += list(np.array(classes)[pe.argmax(1)])
        pooled["conf"] += list(pe.max(1))
        pooled["subject"] += list(te["subject"])
        pooled["label"] += list(te["label"])
        pooled["epoch_id"] += list(te["epoch_id"])
        pooled["probs"] += list(pe)

print("\n=== Per-fold scores (each fold on its own held-out subjects) ===")
display(pd.DataFrame(rows))

y_all = np.array(pooled["y"]); p_all = np.array(pooled["pred"]); c_all = np.array(pooled["conf"])
print("\n=== Pooled CV score (every subject predicted once, by its held-out fold) ===")
print(f"accuracy    : {accuracy_score(y_all, p_all):.1%}")
print(f"macro-F1    : {f1_score(y_all, p_all, average='macro'):.3f}")
print(f"mean conf   : {c_all.mean():.3f}   |   min conf: {c_all.min():.3f}")
print("\n" + classification_report(y_all, p_all, digits=3))

ConfusionMatrixDisplay(
    confusion_matrix(y_all, p_all, labels=classes),
    display_labels=classes).plot()
plt.title("CogniFusion — pooled CV confusion matrix")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(c_all, bins=25, color="steelblue", edgecolor="white")
ax.set_title("Prediction confidence distribution"); ax.set_xlabel("max softmax prob")
plt.tight_layout(); plt.show()

res = pd.DataFrame({"subject": pooled["subject"], "label": pooled["label"],
                    "epoch_id": pooled["epoch_id"], "pred": p_all,
                    "confidence": np.round(c_all, 3)})
for j, cname in enumerate(classes):
    res[f"p_{cname}"] = np.round(np.array(pooled["probs"])[:, j], 3)
res["correct"] = (res["label"] == res["pred"]).astype(int)
subj = res.groupby("subject")["correct"].mean().sort_values()
print("\nPer-subject accuracy (all 30 subjects, each predicted once):")
print(subj.round(3).to_string())
subj.plot.barh(figsize=(6, 5), color="seagreen")
plt.title("Accuracy per subject (pooled CV)"); plt.xlabel("accuracy")
plt.tight_layout(); plt.show()

# --- Sample predictions with full probability vectors ---
# recompute stacked probs per row is heavy; show pooled preds + confidence instead
res["confidence"] = np.round(c_all, 3)
display(res.head(15))

## 10 · Model diagnostics

Beyond accuracy: **ROC-AUC** (class separability), **calibration** (do the
probabilities mean what they say?), and **LightGBM feature importance**
(which fused features the tree model relies on most).

*Requires section 9 to have run (uses its pooled predictions).*

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.preprocessing import label_binarize

# pooled predictions from section 9
y_true = np.array(pooled["y"]); P = np.array(pooled["probs"])
classes_arr = np.array(classes)
Y = label_binarize(y_true, classes=classes_arr)

# ---------- ROC-AUC (one-vs-rest) ----------
auc = roc_auc_score(Y, P, average="macro", multi_class="ovr")
print(f"Macro ROC-AUC (ovr): {auc:.3f}")
for j, cname in enumerate(classes_arr):
    fpr, tpr, _ = roc_curve(Y[:, j], P[:, j])
    a = roc_auc_score(Y[:, j], P[:, j])
    print(f"  {cname:8s} AUC = {a:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
for j, cname in enumerate(classes_arr):
    fpr, tpr, _ = roc_curve(Y[:, j], P[:, j])
    ax.plot(fpr, tpr, label=f"{cname} (AUC={roc_auc_score(Y[:,j],P[:,j]):.3f})")
ax.plot([0,1],[0,1],"k--",lw=0.8); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title("ROC curves — one-vs-rest"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# ---------- Calibration (reliability diagram + ECE) ----------
conf = P.max(1); correct = (y_true == classes_arr[P.argmax(1)]).astype(float)
bins = np.linspace(0, 1, 11)
ece = 0.0
fig, ax = plt.subplots(figsize=(6, 5))
xs, ys = [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf >= lo) & (conf < hi)
    if m.sum() == 0: continue
    acc_bin, conf_bin = correct[m].mean(), conf[m].mean()
    xs.append(conf_bin); ys.append(acc_bin)
    ece += m.sum()/len(conf) * abs(acc_bin - conf_bin)
ax.plot([0,1],[0,1],"k--",lw=0.8,label="perfect")
ax.plot(xs, ys, "o-", color="steelblue", label=f"model (ECE={ece:.3f})")
ax.set_xlabel("mean confidence"); ax.set_ylabel("accuracy")
ax.set_title("Calibration — reliability diagram"); ax.legend(); plt.tight_layout(); plt.show()
print(f"Expected Calibration Error: {ece:.3f}  |  Brier score: {brier_score_loss(y_true, P, labels=classes_arr):.3f}")

# ---------- LightGBM feature importance (averaged over folds) ----------
imp = np.zeros(len(eeg_cols) + len(eye_cols))
n_lgb = 0
colnames = list(eeg_cols) + list(eye_cols)
for fold in bundle["folds"]:
    if fold.get("lgb") is not None:
        imp += fold["lgb"].feature_importances_
        n_lgb += 1
if n_lgb:
    imp /= n_lgb
    order = np.argsort(imp)[::-1][:15]
    pd.Series(imp[order], index=[colnames[i] for i in order]).plot.barh(figsize=(9,5), color="seagreen")
    plt.title(f"Top-15 LightGBM feature importances (mean over {n_lgb} folds)")
    plt.xlabel("importance"); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()
else:
    print("No LightGBM models in checkpoint (train without --use-gbm to skip)")

# ---------- Error analysis: where does the model fail? ----------
res2 = pd.DataFrame({"y": y_true, "p": p_all})
res2["pred"] = classes_arr[P.argmax(1)]
res2["correct"] = (res2["y"] == res2["pred"]).astype(int)
acc_by_class = res2.groupby("y")["correct"].mean()
print("Accuracy by true class:")
print(acc_by_class.round(3).to_string())
acc_by_class.plot.bar(figsize=(5,3), color="steelblue")
plt.title("Per-class accuracy"); plt.ylabel("accuracy"); plt.ylim(0,1); plt.tight_layout(); plt.show()

# misclassified samples: show confidence distribution of errors
errs = res2[res2["correct"]==0]
print(f"Errors: {len(errs)}/{len(res2)} ({len(errs)/len(res2):.1%})")
if len(errs):
    print("Mean confidence on errors:", round(float(P[res2["correct"].values==0].max(1).mean()),3))
    print("Confusion pairs (true -> pred):")
    print(errs.groupby(["y","pred"]).size().to_string())


## 10b · Ablation — how much does each modality contribute?

Quick sklearn baselines on the **same subject-wise holdout**: RandomForest on
EEG-only, eye-only, and fused features. On synthetic data all three saturate —
the interesting comparison is on the **real dataset** (section 12), where eye
features typically dominate.

In [ ]:
import sys; sys.path.insert(0, "src")
import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from train import subject_wise_split, adaptive_subject_norm, coerce_numeric, make_norm_key

# works on whichever feature table was built last (synthetic or real)
df_a = pd.read_parquet("data/processed/features.parquet"
                       if Path("data/processed/features.parquet").exists()
                       else "data/processed/features_real.parquet")
df_a["norm_key"] = make_norm_key(df_a)
tr_a, te_a = subject_wise_split(df_a, test_frac=0.25, seed=42)
eeg_cols = [c for c in df_a.columns if c.startswith("eeg__")]
eye_cols = [c for c in df_a.columns if c.startswith("eye__")]

rows = []
for name, cols in [("EEG-only", eeg_cols), ("Eye-only", eye_cols), ("Fused", eeg_cols + eye_cols)]:
    Xtr = coerce_numeric(tr_a, cols); Xte = coerce_numeric(te_a, cols)
    trn, _ = adaptive_subject_norm(Xtr, tr_a["norm_key"])
    ten, _ = adaptive_subject_norm(Xte, te_a["norm_key"])
    m = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
    m.fit(trn.fillna(0), tr_a["label"])
    p = m.predict(ten.fillna(0))
    rows.append({"features": name, "n_cols": len(cols),
                 "accuracy": round(accuracy_score(te_a["label"], p), 4),
                 "macro_f1": round(f1_score(te_a["label"], p, average="macro"), 4)})
abl = pd.DataFrame(rows)
display(abl)
abl.set_index("features")[["accuracy","macro_f1"]].plot.bar(figsize=(7,4))
plt.title("Modality ablation — RandomForest baseline"); plt.ylabel("score"); plt.ylim(0,1)
plt.tight_layout(); plt.show()

## 11 · Data management & persistence

`src/data_management.py` is the same layer the live pipeline uses:

- **`DataLogger`** — SQLite database (`data/logs.db`) with `epochs`, `sessions`,
  `alerts` tables; `query_epochs()` / `get_statistics()` for analysis
- **`SessionManager`** — JSON session recordings under `sessions/` with
  epoch-level playback data

Below we log a sample of the held-out predictions exactly as the pipeline would,
then read them back. In Colab the DB is ephemeral — download `data/logs.db`
or export to CSV/JSON to keep it.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, "src")
from data_management import DataLogger, SessionManager

logger = DataLogger(log_dir="logs", db_path="data/logs.db")
sessions = SessionManager(storage_dir="sessions")

# --- record a "live" session from the pooled CV predictions ---
sid = sessions.create_session("colab_eval", {"source": "notebook", "model": "fusion_model.pt"})
# reuse section-9 results if they exist, else synthesize demo rows
try:
    sample = res.sample(min(30, len(res)), random_state=0)
except NameError:
    import pandas as _pd
    sample = _pd.DataFrame({
        "subject": ["S00"]*10 + ["S01"]*10 + ["S02"]*10,
        "label":   ["low"]*10 + ["medium"]*10 + ["high"]*10,
        "epoch_id": range(30),
        "pred":    ["low"]*10 + ["medium"]*10 + ["high"]*10,
        "confidence": [0.8]*30})

for _, r in sample.iterrows():
    epoch_data = {
        "timestamp": f"2026-11-22T10:{int(r['epoch_id'])//60:02d}:{int(r['epoch_id'])%60:02d}",
        "prediction": r["pred"],
        "confidence": float(r["confidence"]),
        "eeg_trust": 0.52, "eye_trust": 0.48,
        "processing_time": 0.12,
        "features": {"note": f"subject={r['subject']}", "true_label": r["label"]},
    }
    sessions.add_epoch_to_session(sid, epoch_data)
    logger.log_epoch(epoch_data, session_id=sid)

sessions.close_session(sid)
logger.log_session({
    "session_id": sid, "user_id": "colab_eval",
    "start_time": "2026-11-22T10:00:00", "end_time": "2026-11-22T10:30:00",
    "epoch_count": len(sample), "metadata": {"source": "notebook"},
})
logger.log_alert({"type": "info", "severity": "low",
                  "message": "Notebook evaluation session logged"}, session_id=sid)

print("=== Database contents ===")
db = logger.query_epochs()
print(f"{len(db)} epoch rows in SQLite")
display(db[["timestamp", "prediction", "confidence", "eeg_trust", "eye_trust"]].head(10))
print("Statistics:", json.dumps(logger.get_statistics(), indent=2))

# --- session playback file ---
rec = sessions.get_session(sid)
print("Session recording:", rec["session_id"], "-", rec["epoch_count"], "epochs, status:", rec["status"])
Path("sessions").mkdir(exist_ok=True)


In [ ]:
# Export the logged epochs to CSV / JSON for reuse outside Colab
out = logger.query_epochs()
out.to_csv("data/processed/eval_epochs.csv", index=False)
out.to_json("data/processed/eval_epochs.json", orient="records", indent=2)
print("Exported:", "data/processed/eval_epochs.csv", "and .json")

# Optional: download the SQLite DB itself in Colab
try:
    from google.colab import files
    files.download("data/logs.db")
    print("logs.db download started")
except ImportError:
    print("Not in Colab — DB saved at data/logs.db")

# ---------- Session-log timeline: confidence over logged epochs ----------
import matplotlib.pyplot as plt
db = logger.query_epochs().reset_index(drop=True)
if len(db):
    fig, ax = plt.subplots(figsize=(10,3))
    colors = {"low":"seagreen","medium":"goldenrod","high":"tomato"}
    for i,(idx,row) in enumerate(db.iterrows()):
        ax.scatter(idx, row["confidence"], c=colors.get(row["prediction"],"gray"), s=30)
    ax.set_title("Logged session — prediction confidence over epochs")
    ax.set_xlabel("epoch #"); ax.set_ylabel("confidence")
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([0],[0],marker="o",ls="",color=colors[k],label=k) for k in colors], fontsize=8)
    plt.tight_layout(); plt.show()
    print(f"Plotted {len(db)} logged epochs")


## 12 · Real dataset (optional) — 113 participants

The synthetic pipeline above is for demo. For the **real benchmark**, download the
public *Consumer-Grade EEG-based Eye Tracking* dataset (Zenodo, 1.5 GB CSV:
Muse EEG @256 Hz + webcam gaze, 116 sessions, saccade/smooth-pursuit tasks at two
difficulty levels) and rebuild features from it.

> ⏳ Feature extraction over ~10k real epochs takes **hours on CPU** — run this
> section only if you have time, or run it once and reuse `features_real.parquet`.

In [ ]:
# Download the real dataset (~1.5 GB) from Zenodo
import os
if not os.path.exists("data/raw/dataset.csv"):
    !wget -q --show-progress "https://zenodo.org/api/records/14860668/files/dataset.csv/content" -O data/raw/dataset.csv
else:
    print("dataset.csv already present")
!ls -lh data/raw/dataset.csv

In [ ]:
# Build fused features from real data (long step — prints progress every 1000 epochs)
import sys; sys.path.insert(0, "src")
from pathlib import Path
FEAT_REAL = Path("data/processed/features_real.parquet")
if FEAT_REAL.exists():
    print("features_real.parquet exists — skipping")
    import pandas as pd
    df_real = pd.read_parquet(FEAT_REAL)
else:
    from build_dataset_real import build
    df_real = build()
df_real.shape

In [ ]:
# Train on the real dataset — subject-wise 5-fold CV for honest generalization
!python src/train.py --features data/processed/features_real.parquet                     --epochs 200 --cv 5 --use-gbm                     --out models/fusion_model_real

## 13 · Next steps

- **Real data**: `python src/build_dataset_real.py` builds the fused table from the
  1.5 GB real Muse-EEG + webcam dataset (`data/raw/dataset.csv`, 113 participants —
  see the repo README), then train with
  `--features data/processed/features_real.parquet --cv 5`.
- **Live demo**: `python src/realtime_pipeline.py --eeg synthetic --eye synthetic`
  locally, or `streamlit run dashboard.py` for the monitoring dashboard. Webcam
  eye-tracking works in Colab (`src/webcam_eyetracker.py`); EEG needs Muse
  hardware (BrainFlow).
- **Full training**: `--epochs 300 --cv 5` for the production-grade subject-wise CV.

## Limitations

- Synthetic data is generated from known class-conditional patterns — scores here
  are an upper bound, not a real-world guarantee.
- `medium` is intentionally the ambiguous class (partial overlap in the generator);
  expect lower F1 there than on `low`/`high`.
- Single-epoch inference in the notebook uses the holdout split; live deployment
  uses session-adaptive calibration over a rolling window instead.

## 14 · Run summary

Compact recap of everything the notebook produced.

In [ ]:
import torch, json
ckpt = torch.load("models/fusion_model.pt", map_location="cpu", weights_only=False)
print("="*55)
print("COGNIFUSION — NOTEBOOK RUN SUMMARY")
print("="*55)
print(f"Dataset      : {df.shape[0]} epochs, {df['subject'].nunique()} subjects, "
      f"{df['label'].nunique()} classes ({len(df.columns)-3} features)")
print(f"Model        : {len(ckpt['folds'])}-fold ensemble "
      f"({'NN+GBM' if ckpt['folds'][0].get('lgb_model') else 'NN-only'})")
print(f"CV accuracy  : {ckpt['cv_summary']['mean_accuracy']:.1%} "
      f"(macro-F1 {ckpt['cv_summary']['mean_macro_f1']:.3f})")
try:
    print(f"Pooled eval  : {accuracy_score(y_true, p_all):.1%} on "
          f"{len(y_true)} held-out epochs")
    print(f"ROC-AUC      : {auc:.3f}   ECE: {ece:.3f}")
except NameError:
    print("Pooled eval  : (run sections 9-10 first)")
try:
    print(f"DB logged    : {logger.get_statistics()}")
except NameError:
    print("DB logged    : (run section 11 first)")
print("="*55)